## 1 — Imports

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.embeddings import Embeddings
from pathlib import Path
from langchain_pinecone import PineconeVectorStore
from langchain_community.document_loaders import TextLoader
import glob
from openai import OpenAI
import os
from modules.embeddings.nvidia_embeddings import NVIDIAEmbeddings

# Sparse Embeddings
from sentence_transformers import SparseEncoder

/tmp/ipykernel_525206/1528027618.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


## 2 — Config

In [6]:
NVIDIA_API_KEY=os.environ.get("NVIDIA_API_KEY")
PINECONE_API_KEY =os.environ.get("PINECONE_API_KEY")

In [7]:
index_name=os.environ.get("PINECONE_INDEX_NAME")

In [ ]:
print(index_name)

## 3 — Load documents

In [5]:
output_path = "../../data/parsed"
pages_path = glob.glob(f"{output_path}/*.md")

## 4 — Split

In [6]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150
)

docs = []

for page in pages_path:
    text_loader = TextLoader(
        page,
        encoding="utf-8"
    )

    docs.extend(text_loader.load())

chunks = splitter.split_documents(docs)

page_content='# Attention Is All You Need

Ashish Vaswani∗ Google Brain avaswani@google.com

Noam Shazeer∗ Google Brain noam@google.com

Niki Parmar∗   
Google Research   
nikip@google.com

Jakob Uszkoreit∗ Google Research usz@google.com

Llion Jones∗ Google Research llion@google.com

Aidan N. Gomez∗ † University of Toronto aidan@cs.toronto.edu

Łukasz Kaiser∗ Google Brain lukaszkaiser@google.com

Illia Polosukhin∗ ‡illia.polosukhin@gmail.com

## Abstract' metadata={'source': '../data/parsed/page_001.md'}


## 5 — Embeddings

# Sprase EMbedding

In [4]:
model = SparseEncoder("naver/splade-cocondenser-ensembledistil")



Loading weights: 100%|██████████| 204/204 [00:00<00:00, 9689.25it/s]


In [13]:


texts = [chunk.page_content for chunk in chunks]

# for content in range(len(chunks)):
#    embeddings.append(model.encode(chunks[content].page_content))

embeddings = model.encode(texts, batch_size=32, show_progress_bar=True)

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Batches: 100%|██████████| 2/2 [00:01<00:00,  1.20it/s]


In [14]:
embeddings

tensor(indices=tensor([[    0,     0,     0,  ...,    52,    52,    52],
                       [ 1001,  1030,  1105,  ..., 28991, 29214, 29264]]),
       values=tensor([1.3098, 1.0775, 0.8064,  ..., 0.0625, 0.2137, 0.6160]),
       device='cuda:0', size=(53, 30522), nnz=8514, layout=torch.sparse_coo)

In [15]:
# Convert to a format ready for Vector Databases
sparse_embeddings_dict = []

# Move tensor to CPU and convert to a standard SciPy CSR matrix for easy row iteration
csr_matrix = embeddings.to_sparse_csr() 

for i in range(csr_matrix.shape[0]):
    row = csr_matrix[i]
    # Extract non-zero token IDs and their weights
    indices = row.col_indices().tolist()
    values = row.values().tolist()
    
    sparse_embeddings_dict.append({
        "indices": indices,
        "values": values
    })

RuntimeError: col_indices expected sparse row compressed tensor layout but got Sparse